# Travel Story Generator — Marketplace Listing Edition
### AI Content Generator for Flight Centre Influencer Marketplace

**What this does:**
Given a trip from `trip_plans_v2.csv`, generates a polished **marketplace listing description**
that customers see when browsing packages on the Flight Centre platform.

**Key changes from previous version:**
- **Single output style only**: `marketplace_listing` — the customer-facing package description.
  Other styles (Instagram caption, story, short post) removed. The marketplace context requires
  a consistent brand tone.
- **Longer output target**: 150–250 words, suitable for a marketplace card customers browse
  before purchasing.
- **No external references**: The LLM never mentions competitor platforms, external booking sites,
  or third-party URLs. All content uses only Flight Centre inventory data.
- **Current input: database only** (`trip_plans_v2.csv`). See Section 10 for the Phase 2
  pre-submission flow using local JSON (draft data before database write).

**Architecture:**
```
trip_plans_v2.csv  ←  current Phase 1 input (verified, submitted trips only)
   ↓  Select trip by ID or city/vibe
   ↓  Build Trip Context (hotel, days, activities, vibe, season, description)
   ↓  Single LLM call
        ├─ System: marketplace copywriter persona + style rules
        └─ User:   trip context (no retrieval needed — CSV has all data)
   ↓  Output: 150–250 word marketplace listing paragraph

Phase 2 planned:
local_json (draft builder state)  ←  pre-submission, before admin approval
   ↓  Same pipeline, input from frontend state instead of database
```


## Section 1 — Install & Import

In [1]:
# !pip install anthropic pandas google-genai --quiet

In [2]:
import pandas as pd
import json
import re
import os
import random

print('✅ All imports successful')


✅ All imports successful


## Section 2 — LLM Setup

In [3]:
# !pip install google-genai --quiet --upgrade

In [4]:
import os
os.environ["GEMINI_API_KEY"] = "YOUR API KEY"

In [5]:
# ── Option A: Anthropic Claude ────────────────────────────────────
# import anthropic
# ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY","your-api-key-here")
# llm_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
# LLM_MODEL = "claude-sonnet-4-6"
# LLM_PROVIDER = "anthropic"

# ── Option C: Google Gemini (current) ─────────────────────────────
from google import genai

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("GEMINI_API_KEY not found")

client = genai.Client(api_key=API_KEY)
LLM_MODEL    = "gemini-2.5-flash"
LLM_PROVIDER = "gemini"

print(f"✅ LLM provider: {LLM_PROVIDER} | Model: {LLM_MODEL}")


✅ LLM provider: gemini | Model: gemini-2.5-flash


## Section 3 — Load Trip Plans Dataset

`trip_plans_v2.csv` schema (one row = one trip × one day × one slot):

| Column | Description |
|---|---|
| `trip_id` | Unique trip identifier |
| `trip_name` | Human-readable trip name |
| `city` / `country` | Destination |
| `vibe` | Mood tags (e.g., Romantic; Modern; Vibrant) |
| `best_season` | Best time to visit |
| `total_days` | Trip length |
| `hotel_name`, `hotel_stars`, `hotel_room_type`, `hotel_price_per_night_aud` | Accommodation |
| `day_number` / `day_slot` | Day + Morning/Afternoon/Evening |
| `activity_name`, `activity_category`, `activity_price_aud`, `activity_rating` | Activity |
| `description` | Rich activity description |


In [6]:
trips_df = pd.read_csv("trip_plans_v2.csv")

print(f"✅ Loaded {len(trips_df)} rows — {trips_df['trip_id'].nunique()} unique trips")

trip_summary = (
    trips_df
    .groupby(['trip_id','trip_name','city','country','vibe','best_season','total_days'])
    .size().reset_index(name='slots').head(15)
)
print("\nSample trips:")
print(trip_summary[['trip_id','trip_name','city','total_days','vibe']].to_string(index=False))


✅ Loaded 5718 rows — 240 unique trips

Sample trips:
     trip_id                               trip_name      city  total_days                                vibe
INT-TP-00001            The Best of Tokyo in 11 Days     Tokyo          11       Bustling; Modern; Traditional
INT-TP-00002     Cultural Immersion: 9 Days in Paris     Paris           9            Romantic; Artistic; Chic
INT-TP-00003            Brisbane on a Budget: 9 Days  Brisbane           9           Outdoorsy; Relaxed; Sunny
INT-TP-00004   Discover Berlin: 14-Day Explorer Pack    Berlin          14            Edgy; Creative; Historic
INT-TP-00005         Food Lover's 9-Day Dubai Escape     Dubai           9      Luxurious; Futuristic; Vibrant
INT-TP-00006 Discover Singapore: 6-Day Explorer Pack Singapore           6        Modern; Multicultural; Clean
INT-TP-00007      Ultimate London Experience: 5 Days    London           5    Historic; Cosmopolitan; Eclectic
INT-TP-00008     Food Lover's 10-Day New York Escape  New Y

## Section 4 — Trip Context Builder

In [7]:
def get_trip_by_id(trip_id: str) -> pd.DataFrame:
    df = trips_df[trips_df['trip_id'] == trip_id].copy()
    if df.empty:
        raise ValueError(f"Trip '{trip_id}' not found.")
    return df.sort_values(['day_number','day_slot'])


def get_random_trip(city: str | None = None, vibe_keyword: str | None = None) -> pd.DataFrame:
    df = trips_df.copy()
    if city:
        df = df[df['city'].str.lower() == city.lower()]
    if vibe_keyword:
        df = df[df['vibe'].str.lower().str.contains(vibe_keyword.lower(), na=False)]
    if df.empty:
        raise ValueError(f"No trip found for city={city}, vibe={vibe_keyword}")
    return get_trip_by_id(random.choice(df['trip_id'].unique()))


def build_trip_context(trip_rows: pd.DataFrame) -> str:
    """Convert trip rows to structured text context for the LLM."""
    r0 = trip_rows.iloc[0]
    meta = (
        f"Trip name    : {r0['trip_name']}\n"
        f"Destination  : {r0['city']}, {r0['country']}\n"
        f"Duration     : {r0['total_days']} days\n"
        f"Vibe         : {r0['vibe']}\n"
        f"Best season  : {r0['best_season']}\n"
        f"Suitable for : {r0.get('suitable_for','everyone')}\n"
    )
    hotel = (
        f"Hotel        : {r0['hotel_name']} "
        f"({r0['hotel_stars']}-star, {r0['hotel_room_type']}, "
        f"AUD${r0['hotel_price_per_night_aud']}/night)\n"
    )

    # Estimate total cost
    total_activity = trip_rows['activity_price_aud'].sum()
    hotel_total    = float(r0['hotel_price_per_night_aud']) * float(r0['total_days'])
    total_est      = total_activity + hotel_total

    days_text = []
    for day_num in sorted(trip_rows['day_number'].unique()):
        day_rows  = trip_rows[trip_rows['day_number'] == day_num]
        day_lines = [f"  Day {day_num}:"]
        for _, row in day_rows.iterrows():
            slot = row.get('day_slot','')
            act  = row.get('activity_name','')
            cat  = row.get('activity_category','')
            dur  = row.get('activity_duration_hours','')
            desc = str(row.get('description',''))[:150].strip()
            line = f"    [{slot}] {act} ({cat}, {dur})"
            if desc:
                line += f"\n           {desc}..."
            day_lines.append(line)
        days_text.append("\n".join(day_lines))

    itinerary_block = "\n".join(days_text[:6])
    if float(r0['total_days']) > 6:
        itinerary_block += f"\n  ... (+{int(r0['total_days'])-6} more days)"

    return f"""=== TRIP DATA ===
{meta}{hotel}Estimated cost: AUD${total_est:.0f} total

=== ITINERARY ===
{itinerary_block}
"""

print('✅ Trip context builder ready')


✅ Trip context builder ready


## Section 5 — Marketplace Listing Style

**Single style only**: `marketplace_listing`

This is the customer-facing description shown on a travel package card in the Flight Centre
Influencer Marketplace. Customers browse this text before deciding to purchase the package.

Target: 150–250 words. First-person narrative. Focus on experience and vibe.
No prices. No external references.


In [8]:
MARKETPLACE_STYLE = {
    "label":       "Marketplace Package Listing",
    "tone":        "evocative, aspirational, first-person, warm",
    "length":      "150 to 250 words",
    "description": (
        "A polished travel package description for the Flight Centre Influencer Marketplace. "
        "Written as a compelling first-person narrative that conveys the experience, destination "
        "character, and emotional appeal of the package. Suitable for customers browsing packages "
        "before purchase. Reads like a travel journalist describing their own journey."
    ),
}

print(f"✅ Style: {MARKETPLACE_STYLE['label']}")
print(f"   Target length: {MARKETPLACE_STYLE['length']}")


✅ Style: Marketplace Package Listing
   Target length: 150 to 250 words


## Section 6 — Prompt Builder

In [9]:
def build_system_prompt() -> str:
    s = MARKETPLACE_STYLE
    return f"""You are a professional travel copywriter for Flight Centre's Influencer Marketplace.
Your job: read a structured trip itinerary and write one marketplace package description.

STYLE: {s['label']}
TONE: {s['tone']}
LENGTH: {s['length']}
DESCRIPTION: {s['description']}

RULES:
- Write in first person ("I", "we") as if you experienced this trip
- Open with the destination's character — landscape, atmosphere, feeling
- Weave in the vibe tags and seasonal details naturally
- Highlight 3-4 specific activities by name, emphasising what makes them memorable
- Mention the hotel if it adds to the story (higher star ratings are worth noting)
- Close with an emotional invitation — why a traveller would choose this package
- 150 to 250 words. Count your words before finishing.
- Do NOT mention prices, AUD values, or cost
- Do NOT reference any external booking platform, competitor site, or third-party URL
- Do NOT use generic filler phrases ("experience the magic of", "unforgettable adventure")
- Use specific sensory details from the activity descriptions provided
- Output ONLY the final listing text — no preamble, no title, no labels
"""


def build_user_prompt(trip_context: str, custom_note: str = "") -> str:
    custom_block = f"\nADDITIONAL INSTRUCTION: {custom_note}" if custom_note else ""
    return f"""Write the marketplace listing for this trip.

{trip_context}
{custom_block}
Output only the listing text. 150-250 words. First person.
"""

print('✅ Prompt builder ready')


✅ Prompt builder ready


## Section 7 — LLM Caller

In [10]:
def call_story_llm(system_prompt: str, user_prompt: str) -> str:
    if LLM_PROVIDER == "anthropic":
        response = llm_client.messages.create(
            model=LLM_MODEL, max_tokens=700, system=system_prompt,
            messages=[{"role":"user","content":user_prompt}])
        return response.content[0].text.strip()

    elif LLM_PROVIDER == "openai":
        response = llm_client.chat.completions.create(
            model=LLM_MODEL, max_tokens=700,
            messages=[{"role":"system","content":system_prompt},
                      {"role":"user","content":user_prompt}])
        return response.choices[0].message.content.strip()

    elif LLM_PROVIDER == "gemini":
        full_prompt = "[SYSTEM]\n" + system_prompt + "\n\n[TRIP DATA]\n" + user_prompt
        response = client.models.generate_content(model=LLM_MODEL, contents=full_prompt)
        return response.text.strip()

    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")

print('✅ LLM caller ready')


✅ LLM caller ready


## Section 8 — Main Generator Function

In [11]:
def validate_listing(text: str) -> tuple[bool, list[str]]:
    """
    Basic quality checks on generated listing.
    Returns (passed, issues_list).
    """
    issues = []
    word_count = len(text.split())
    if word_count < 120:
        issues.append(f"Too short: {word_count} words (minimum 120)")
    if word_count > 280:
        issues.append(f"Too long: {word_count} words (maximum 280)")

    # Check for banned references
    banned = ['booking.com','tripadvisor','expedia','airbnb','google travel',
              'klook','viator','get your guide','getyourguide']
    for b in banned:
        if b in text.lower():
            issues.append(f"Contains external reference: '{b}'")

    # Check for price mentions
    if re.search(r'AUD\$|\$[0-9]|per night|cost', text, re.IGNORECASE):
        issues.append("Contains price/cost reference — remove for marketplace listing")

    return len(issues) == 0, issues


def generate_listing(
    trip_id: str | None     = None,
    city: str | None         = None,
    vibe_keyword: str | None = None,
    custom_note: str         = "",
    verbose: bool            = True,
    max_retries: int         = 3,
) -> tuple[str, pd.DataFrame]:
    """
    Generate a marketplace listing for a trip.

    Args:
        trip_id      : Specific trip (e.g. 'INT-TP-00001')
        city         : Filter by city if no trip_id
        vibe_keyword : Filter by vibe
        custom_note  : Extra instruction for the LLM
        verbose      : Print progress
        max_retries  : Retry on quality validation failure

    Returns:
        (listing_text, trip_rows)
    """
    if trip_id:
        trip_rows = get_trip_by_id(trip_id)
    else:
        trip_rows = get_random_trip(city=city, vibe_keyword=vibe_keyword)

    r0 = trip_rows.iloc[0]
    if verbose:
        print(f"\n{'='*60}")
        print(f"Generating listing: {r0['trip_name']}")
        print(f"  {r0['city']}, {r0['country']} | {r0['total_days']} days | {r0['vibe'][:50]}")
        print('='*60)

    trip_context  = build_trip_context(trip_rows)
    system_prompt = build_system_prompt()
    user_prompt   = build_user_prompt(trip_context, custom_note)

    for attempt in range(1, max_retries + 1):
        if verbose: print(f"[LLM] Calling {LLM_MODEL} (attempt {attempt})...")
        listing = call_story_llm(system_prompt, user_prompt)
        passed, issues = validate_listing(listing)
        if passed:
            if verbose:
                word_count = len(listing.split())
                print(f"[OK]  {word_count} words, all quality checks passed")
            return listing, trip_rows
        else:
            if verbose:
                print(f"[RETRY] Issues: {issues}")
            # Add guidance for retry
            user_prompt += f"\n\nPrevious attempt failed quality checks: {issues}. Please fix these in your next response."

    if verbose:
        print("[WARN] Max retries reached, returning last attempt")
    return listing, trip_rows

print('✅ generate_listing() ready')


✅ generate_listing() ready


## Section 9 — Output Renderer

In [12]:
from IPython.display import display, HTML

def render_listing(listing: str, trip_rows: pd.DataFrame) -> None:
    r0 = trip_rows.iloc[0]
    listing_html = listing.replace("\n","<br>")
    word_count   = len(listing.split())

    html = f"""
<div style='max-width:680px;border:1px solid #ddd;border-radius:10px;overflow:hidden;font-family:Georgia,serif;margin:12px 0'>
  <div style='background:#1F4E79;padding:14px 20px;color:white;display:flex;justify-content:space-between;align-items:center'>
    <div>
      <div style='font-size:18px;font-weight:bold'>Flight Centre — Package Listing</div>
      <div style='font-size:12px;opacity:0.8;margin-top:2px'>{r0['trip_name']}</div>
    </div>
    <div style='font-size:12px;opacity:0.8;text-align:right'>
      {r0['city']}, {r0['country']}<br>{r0['total_days']} days
    </div>
  </div>
  <div style='background:#EBF3FB;padding:8px 18px;font-size:12px;color:#555;border-bottom:1px solid #ddd'>
    <b>Vibe:</b> {r0['vibe'][:60]} &nbsp;·&nbsp; <b>Best season:</b> {r0['best_season']}
    &nbsp;·&nbsp; <b>Word count:</b> {word_count}
  </div>
  <div style='padding:20px 24px;font-size:15px;line-height:1.8;color:#222;background:white'>
    {listing_html}
  </div>
  <div style='background:#f8f8f8;padding:8px 18px;font-size:11px;color:#aaa;border-top:1px solid #eee;text-align:right'>
    Generated by Travel Story Generator · {LLM_MODEL} · Flight Centre Influencer Marketplace
  </div>
</div>
"""
    display(HTML(html))

print('✅ Renderer ready')


✅ Renderer ready


## Section 10 — Demo: Generate Marketplace Listings

In [13]:
# Demo 1: Specific trip by ID
listing1, rows1 = generate_listing(trip_id="INT-TP-00001")
render_listing(listing1, rows1)



Generating listing: The Best of Tokyo in 11 Days
  Tokyo, Japan | 11 days | Bustling; Modern; Traditional
[LLM] Calling gemini-2.5-flash (attempt 1)...
[OK]  213 words, all quality checks passed


In [14]:
# Demo 2: Random trip from Tokyo
listing2, rows2 = generate_listing(city="Tokyo")
render_listing(listing2, rows2)



Generating listing: Tokyo on a Budget: 3 Days
  Tokyo, Japan | 3 days | Bustling; Modern; Traditional
[LLM] Calling gemini-2.5-flash (attempt 1)...
[OK]  215 words, all quality checks passed


In [15]:
# Demo 3: Romantic trip from Paris
listing3, rows3 = generate_listing(city="Paris", vibe_keyword="romantic")
render_listing(listing3, rows3)



Generating listing: Discover Paris: 7-Day Explorer Pack
  Paris, France | 7 days | Romantic; Artistic; Chic
[LLM] Calling gemini-2.5-flash (attempt 1)...
[RETRY] Issues: ['Contains price/cost reference — remove for marketplace listing']
[LLM] Calling gemini-2.5-flash (attempt 2)...


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 35.961087533s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [ ]:
# Demo 4: Luxury Dubai trip with custom instruction
listing4, rows4 = generate_listing(
    city="Dubai", vibe_keyword="Luxurious",
    custom_note="Emphasise the contrast between desert landscapes and modern architecture."
)
render_listing(listing4, rows4)


## Section 11 — Batch Generation

Generate one listing per city across multiple destinations.


In [ ]:
CITIES = ["Tokyo", "Paris", "Dubai", "Bali", "Sydney", "Seoul", "London"]

batch = []
for city in CITIES:
    try:
        listing, rows = generate_listing(city=city, verbose=False)
        render_listing(listing, rows)
        batch.append({"city": city, "words": len(listing.split()), "preview": listing[:100]+"..."})
    except ValueError as e:
        print(f"Skipped {city}: {e}")

print(f"\nBatch complete: {len(batch)} listings generated")


## Section 12 — Phase 2: Pre-Submission Input (Local JSON)

In Phase 2, the content generator will accept draft itinerary data from the frontend
builder **before** it is written to the database. This allows influencers to preview
their marketplace description during the package creation flow.

The input changes from `trip_plans_v2.csv` to a **local JSON payload** constructed
from the frontend React state — data the influencer has entered but has not yet
submitted for administrator approval.


In [ ]:
# ── Phase 2 draft input format (local JSON, not database) ──────────────
# This function accepts the JSON payload that the frontend will send.
# The schema mirrors trip_plans_v2 but comes from the builder state, not the DB.

def generate_listing_from_draft(draft_json: dict, custom_note: str = "") -> str:
    """
    Generate a marketplace listing from a pre-submission draft payload.
    Input: JSON dict from the frontend builder state (not yet in database).
    Output: listing text for preview before submission.

    Phase 2 usage:
        draft = {
            "trip_name": "My Tokyo Adventure",
            "city": "Tokyo", "country": "Japan",
            "total_days": 5,
            "vibe": "Bustling; Modern",
            "best_season": "Spring",
            "hotel_name": "Park Hyatt Tokyo", "hotel_stars": 5,
            "hotel_room_type": "Deluxe", "hotel_price_per_night_aud": 450,
            "days": [
                {"day_number": 1, "slots": [
                    {"day_slot": "Morning", "activity_name": "Tsukiji Outer Market Tour",
                     "activity_category": "Food", "activity_duration_hours": "2 hrs",
                     "description": "Sample fresh sushi and street food at Tokyo's famous market..."}
                ]}
            ]
        }
    """
    # Build context from draft JSON (same structure as build_trip_context but from dict)
    meta = (
        f"Trip name    : {draft_json.get('trip_name','Draft Trip')}\n"
        f"Destination  : {draft_json.get('city','')}, {draft_json.get('country','')}\n"
        f"Duration     : {draft_json.get('total_days','')} days\n"
        f"Vibe         : {draft_json.get('vibe','')}\n"
        f"Best season  : {draft_json.get('best_season','')}\n"
    )
    hotel = (
        f"Hotel        : {draft_json.get('hotel_name','')} "
        f"({draft_json.get('hotel_stars','')} star, {draft_json.get('hotel_room_type','')})\n"
    )

    days_text = []
    for day in draft_json.get("days", []):
        day_lines = [f"  Day {day['day_number']}:"]
        for slot in day.get("slots",[]):
            desc = str(slot.get('description',''))[:150].strip()
            line = f"    [{slot.get('day_slot','')}] {slot.get('activity_name','')} ({slot.get('activity_category','')})"
            if desc:
                line += f"\n           {desc}..."
            day_lines.append(line)
        days_text.append("\n".join(day_lines))

    trip_context = f"=== DRAFT TRIP DATA (PRE-SUBMISSION) ===\n{meta}{hotel}\n=== DRAFT ITINERARY ===\n" + "\n".join(days_text)

    system_prompt = build_system_prompt()
    user_prompt   = build_user_prompt(trip_context, custom_note)
    return call_story_llm(system_prompt, user_prompt)


print("✅ Phase 2 draft input function ready")
print("   Usage: generate_listing_from_draft(draft_json) — accepts frontend JSON payload")
print("   Note: This function uses data NOT yet in the database (pre-submission draft).")
